# NFL Data Processing — Build Lookup Tables for RL Environment

Reads `plays.csv` and `defensive_formations.csv`, produces `lookup_table.json` with:
1. **Play outcomes** — raw yardsGained arrays per situation bucket
2. **Turnover rates** — interception/fumble probability per bucket
3. **Defense distributions** — defensive formation frequencies per situation

In [ ]:
# Cell 1: Load & Merge Data
import pandas as pd
import json
import random

plays = pd.read_csv('data/plays.csv')
def_formations = pd.read_csv('data/defensive_formations.csv')

print(f"Plays: {len(plays)}, Defensive formations: {len(def_formations)}")

# Merge on gameId + playId
df = plays.merge(def_formations[['gameId', 'playId', 'defFormation']], on=['gameId', 'playId'], how='inner')
print(f"After merge: {len(df)} plays ({len(plays) - len(df)} dropped — no defensive formation match)")

# Drop plays with no offense formation
before = len(df)
df = df.dropna(subset=['offenseFormation'])
print(f"Dropped NaN offenseFormation: {before - len(df)} plays")

# Derive playType: run, pass, play_action
df['playType'] = 'run'
df.loc[df['isDropback'] == True, 'playType'] = 'pass'
df.loc[(df['isDropback'] == True) & (df['playAction'] == True), 'playType'] = 'play_action'

print(f"\nPlay type distribution:\n{df['playType'].value_counts()}")

# Drop JUMBO and WILDCAT formations
before = len(df)
df = df[~df['offenseFormation'].isin(['JUMBO', 'WILDCAT'])].copy()
print(f"\nDropped JUMBO/WILDCAT: {before - len(df)} plays removed, {len(df)} remaining")

print(f"\nOffense formations:\n{df['offenseFormation'].value_counts()}")
print(f"\nDefensive formations:\n{df['defFormation'].value_counts()}")

In [ ]:
# Cell 2: Analyze Bucket Sizes
buckets = df.groupby(['offenseFormation', 'playType', 'defFormation']).size().reset_index(name='count')
print(f"Total unique buckets: {len(buckets)}")
print(f"\nBucket size distribution:")
print(buckets['count'].describe())

# Show buckets with fewer than 10 plays
small_buckets = buckets[buckets['count'] < 10].sort_values('count')
print(f"\nBuckets with <10 plays: {len(small_buckets)}")
if len(small_buckets) > 0:
    print(small_buckets.to_string(index=False))

# Identify rare defensive formations to merge
rare_defs = buckets.groupby('defFormation')['count'].sum().sort_values()
print(f"\nTotal plays by defensive formation:")
print(rare_defs.to_string())

# Merge rare defenses into "Other" if they contribute to small buckets
rare_def_names = ['5-3 Heavy', '6-2 Goal Line', '4-4', 'Quarter']
rare_count = df[df['defFormation'].isin(rare_def_names)].shape[0]
print(f"\nRare defenses ({rare_def_names}): {rare_count} total plays")
print("→ Merging these into 'Other' to reduce sparse buckets")

df['defFormation'] = df['defFormation'].replace({name: 'Other' for name in rare_def_names})

# Re-check bucket sizes after merging
buckets2 = df.groupby(['offenseFormation', 'playType', 'defFormation']).size().reset_index(name='count')
small2 = buckets2[buckets2['count'] < 5]
print(f"\nAfter merging rare defenses:")
print(f"  Total buckets: {len(buckets2)}")
print(f"  Buckets with <5 plays: {len(small2)}")
if len(small2) > 0:
    print(small2.to_string(index=False))
print(f"\nDefensive formations now: {df['defFormation'].nunique()}")
print(df['defFormation'].value_counts())

In [ ]:
# Cell 3: Build Lookup Table
lookup = {}

for (formation, play_type, def_formation), group in df.groupby(['offenseFormation', 'playType', 'defFormation']):
    key = f"{formation}|{play_type}|{def_formation}"
    lookup[key] = {
        "yards": group['yardsGained'].tolist(),
        "results": group['passResult'].fillna('none').tolist(),  # 'none' for runs
    }

print(f"Total buckets: {len(lookup)}")
sizes = [len(v['yards']) for v in lookup.values()]
print(f"Bucket sizes — min: {min(sizes)}, max: {max(sizes)}, median: {sorted(sizes)[len(sizes)//2]}")
print(f"Total plays in lookup: {sum(sizes)}")

In [ ]:
# Cell 4: Build Reward Mapping
# Reward logic:
#   passResult == "IN" (interception) → -5
#   passResult == "S" (sack) → -3
#   passResult == "I" (incomplete) → 0
#   Run or complete pass → yardsGained (raw value)

def compute_reward(yards, result):
    if result == 'IN':
        return -5
    elif result == 'S':
        return -3
    elif result == 'I':
        return 0
    else:
        return yards

for key, data in lookup.items():
    rewards = [compute_reward(y, r) for y, r in zip(data['yards'], data['results'])]
    data['rewards'] = rewards

# Verify reward mapping
print("Reward mapping examples:")
for result_type in ['C', 'I', 'S', 'IN', 'R', 'none']:
    for key, data in lookup.items():
        for i, r in enumerate(data['results']):
            if r == result_type:
                print(f"  {result_type}: yards={data['yards'][i]}, reward={data['rewards'][i]}")
                break
        else:
            continue
        break

# Summary stats
all_rewards = [r for data in lookup.values() for r in data['rewards']]
print(f"\nReward distribution:")
print(f"  Mean: {sum(all_rewards)/len(all_rewards):.2f}")
print(f"  Min: {min(all_rewards)}, Max: {max(all_rewards)}")
print(f"  Negative: {sum(1 for r in all_rewards if r < 0)}")
print(f"  Zero: {sum(1 for r in all_rewards if r == 0)}")
print(f"  Positive: {sum(1 for r in all_rewards if r > 0)}")

In [ ]:
# Cell 5: Build Defense Distribution
defense_counts = df['defFormation'].value_counts()
defense_distribution = (defense_counts / defense_counts.sum()).to_dict()

print("Defensive formation distribution (for weighted sampling later):")
for name, prob in sorted(defense_distribution.items(), key=lambda x: -x[1]):
    print(f"  {name}: {prob:.3f} ({defense_counts[name]} plays)")

In [ ]:
# Cell 6: Save & Validate
import os

output = {
    "lookup": lookup,
    "defense_distribution": defense_distribution,
}

with open('data/lookup_table.json', 'w') as f:
    json.dump(output, f)

file_size = os.path.getsize('data/lookup_table.json') / 1024
print(f"Saved data/lookup_table.json ({file_size:.0f} KB)")

# Validate: sample from 5 random buckets
print("\n--- Validation: 10 random samples from 5 random buckets ---")
random.seed(42)
sample_keys = random.sample(list(lookup.keys()), 5)
for key in sample_keys:
    samples = random.choices(lookup[key]['rewards'], k=10)
    print(f"  {key}: {samples}")

# Action space summary
print("\n--- Action Space Summary ---")
action_combos = df.groupby(['offenseFormation', 'playType']).size().reset_index(name='count')
action_combos = action_combos.sort_values('count', ascending=False)
print(action_combos.to_string(index=False))
print(f"\nTotal (formation, playType) combos: {len(action_combos)}")

# Final stats
print(f"\n--- Final Stats ---")
print(f"Total plays in lookup: {sum(len(v['yards']) for v in lookup.values())}")
print(f"Total buckets: {len(lookup)}")
print(f"Offensive formations: {df['offenseFormation'].nunique()} — {sorted(df['offenseFormation'].dropna().unique())}")
print(f"Play types: {sorted(df['playType'].dropna().unique())}")
print(f"Defensive formations: {df['defFormation'].nunique()} — {sorted(df['defFormation'].dropna().unique())}")